In [1]:
import gurobipy as gp
from gurobipy import GRB
import math
import random
#importo la libreria che mi permette di creare un modello matematico,aggiungere vincoli,definire la FO,lanciare l'ottimizzazione e leggerne i risultati
#la classe GRB , che è una parte specifica della libreria di gurobi , contiene tutte le costanti usate per definire il tipo di variabile ed il tipo di 
#ottimizzazione (binaria,intera,continua , max\min)
#import math importa funzioni matematiche o meglio la libreria matematica standard di python
#import random serve per generare numeri casuali relativi ai parametri dell'istanza grande (posizioni casuali\domande variabili)

# GENERAZIONE DEI DATI 

random.seed(42) #imposto il seed del generatore di numeri casuali,serve per garantire riproducibilità dell'istanza e cioè ogni volta che eseguo il codice , i dati generati saranno gli stessi
n_c = 100  #numero dei clienti dell'istanza grande
n_l = 12    #numero dei lockers disponibili ,più coerente rispetto a scenari realistici
customers = list(range(1, n_c + 1))  #creo l'insieme dei clienti , quest'ultimi sono identificati con interi consecutivi
lockers = list(range(n_c + 1, n_c + n_l + 1))  #creo l'insieme dei lockers ,identificati anch'essi con interi consecutivi
nodes = [0] + customers + lockers #creo l'insieme completo dei nodi costituito dal nodo deposito , dai nodi clienti e dai nodi lockers
 
# DEFINISCO LE COORDINATE DEI CLIENTI , DEI LOCKERS E DEL DEPOSITO

coords = {}
for i in customers:
    coords[i] = (
        random.uniform(0, 200),
        random.uniform(0, 200)) #i clienti sono distribuiti in modo casuale in un quadrato 200x200 che rappresenta un'area urbana simulata
 
    
#i lockers invece sono stati posizionati in modo strategico affinchè non siano nè troppo vicini ai clienti nè troppo vicini al deposito

locker_positions = [
    (30, 30), (70, 30), (130, 30), (170, 30),
    (30, 100), (100, 100), (170, 100),
    (30, 170), (70, 170), (130, 170), (170, 170),
    (100, 60)
]     
 
coords[0] = (100, 100) #il deposito è posizionato al centro dell'area
#associo ogni locker alla sua posizione predefinita , si hanno dunque delle posizioni fisse che rappresentano punti di ritiro reali 
for b, pos in zip(lockers, locker_positions): #accoppia l'ID del locker(101-112) con la sua coordinata(x,y)
    coords[b] = pos
#calcolo la distanza euclidea tra tutti i nodi , le quali determinano il costo del routing e la compatibilità cliente-locker
dist = {(i, j): math.sqrt((coords[i][0]-coords[j][0])**2 + (coords[i][1]-coords[j][1])**2) for i in nodes for j in nodes}
 
# PARAMETRI DEL MODELLO

Q = 20             #capacità limitata del veicolo , ora può trasportare 20 pacchi
v_b = 15            #capacità limitata di ogni locker, ora può contenere 15 pacchi
R_b = 40          #raggio di copertura,un cliente può usare un locker solo se dista <= 40
K_max = 10           #numero massimo di veicoli (saturazione estrema) solo 10 veicoli per 100 clienti
alpha1, alpha2 = 1.0, 0.1   #pesi della funzione obiettivo (maggior peso al costo di trasporto/routing piuttosto che al costo di accesso del cliente al locker)
 
R = {b: R_b for b in lockers}  #creo un dizionario che assegna ad ogni locker il suo raggio di copertura
 
a = {}
for i in customers:
    for b in lockers:
        a[i, b] = 1 if dist[i, b] <= R[b] else 0   #la matrice a[i,b] vale 1 se il cliente i si trova entro il raggio del locker b ,0 altrimenti e serve 
        #per il vincolo yib<=aib che impedisce assegnamenti impossibili
 
model = gp.Model("VRP_100_EXACT_VS_HEURISTIC")  #creazione del modello per l'istanza grande
 
# VARIABILI DEL MODELLO E FUNZIONE OBIETTIVO

x = model.addVars(nodes, nodes, vtype=GRB.BINARY, name="x") #variabile binaria xij che vale 1 se il veicolo percorre l'arco(i,j) ,0 altrimenti
y = model.addVars(customers, lockers, vtype=GRB.BINARY, name="y") #variabile binaria yib che vale 1 se il cliente i ritira al locker b,0 altrimenti
f = model.addVars(nodes, nodes, vtype=GRB.CONTINUOUS, name="f") #variabile di flusso continua ,rappresenta la quantità di pacchi trasportati sull'arco (i,j)
 
model.setObjective(
   alpha1* gp.quicksum(dist[i, j] * x[i, j] for i in nodes for j in nodes if i != j) + 
   alpha2* gp.quicksum(dist[i, b]* y[i,b] for i in customers for b in lockers), 
    GRB.MINIMIZE
)
#la funzione obiettivo minimizza la distanza totale percorsa dal veicolo ed il costo di accesso ai locker,pesati rispettivamente da alfa1 ed alfa2 
 
#questo vincolo elimina gli archi che partono e arrivano sullo stesso nodo

for i in nodes:
    model.addConstr(x[i, i] == 0) #impongo x[i,i] = 0 per evitare che il veicolo percorra archi non realistici
    model.addConstr(f[i, i] == 0) #impongo f[i,i] = 0 per impedire che il flusso  circoli su archi inesistenti
 
#questo vincolo garantisce la continuità della rotta , la somma degli archi entranti in un nodo deve essere uguale alla somma degli archi uscenti
#dunque se il veicolo visita un nodo , deve anche lasciarlo evitando nodi isolati o percorsi isolati

for i in nodes:
    model.addConstr(
        gp.quicksum(x[j, i] for j in nodes if j != i) ==
        gp.quicksum(x[i, j] for j in nodes if j != i)
    )
 
#questo vincolo garantisce che ogni cliente sia servito esattamente una volta o dal veicolo o da un locker,la somma degli archi entranti (xji) nel 
#cliente rappresenta la consegna a domicilio , mentre la somma delle variabili yᵢᵇ rappresenta la consegna via locker quindi impongo che la loro 
#somma sia pari a 1

for i in customers:
    model.addConstr(
        gp.quicksum(x[j, i] for j in nodes if j != i) +
        gp.quicksum(y[i, b] for b in lockers)
        == 1
    )
 
#questo vincolo garantisce la compatibilità geografica cliente-locker, la matrice a[i,b] indica se il cliente i si trova entro il raggio di copertura 
#del locker b, impongo poi la disuguaglianza cosicchè il cliente possa essere assegnato ad un locker solo se è effettivamente raggiungibile

for i in customers:
    for b in lockers:
        model.addConstr(y[i, b] <= a[i, b])
 
#questo vincolo garantisce che ogni locker non superi la propria capacità fisica , la variabile yᵢᵇ indica se il cliente i è assegnato al locker b
#la loro somma rappresenta il numero totale di clienti serviti da quel locker dunque impongo che questa somma sia ≤ v_b, cioè la capacità massima
#del locker.(ogni locker può servire al massimo 2 clienti)

for b in lockers:
    model.addConstr(
        gp.quicksum(y[i, b] for i in customers) <= v_b
    )
 
#questo vincolo collega l’assegnamento dei clienti ai locker con il routing del veicolo, quindi se un cliente è assegnato a un locker, la variabile
#yᵢᵇ vale 1 ed in tal caso, il modello impone che almeno un arco entrante nel locker sia attivo, cioè il veicolo deve visitarlo

for i in customers:
    for b in lockers:
        model.addConstr(
            gp.quicksum(x[j, b] for j in nodes if j != b) >= y[i, b]
        )
 
#questo vincolo limita il numero di veicoli che possono partire dal deposito (al max K_max) , la somma delle variabili x0j
#(1 se il veicolo percorre l'arco dal deposito al nodo j) rappresenta il numero di rotte che iniziano dal deposito.

model.addConstr(
    gp.quicksum(x[0, j] for j in nodes if j != 0) <= K_max
)
 
#questo vincolo limita il numero di veicoli che possono rientrare al deposito (al max K_max) , la somma delle variabili xi0
#(1 se il veicolo percorre l'arco dal nodo i al deposito) rappresenta il numero di rotte che terminano al deposito

model.addConstr(
    gp.quicksum(x[i, 0] for i in nodes if i != 0) <= K_max
)
 
#questo vincolo stabilisce il flusso totale in uscita dal deposito, ogni cliente richiede una unità di domanda dunque il deposito deve iniviare
#esattamente 100 unità 

model.addConstr(
    gp.quicksum(f[0, j] for j in nodes if j != 0) == len(customers)
)
 
#questo vincolo è relativo alla conservazione del flusso sui nodi clienti, se il cliente è servito dal veicolo, il nodo consuma una unità di flusso
#se invece è assegnato a un locker, il flusso non deve raggiungere il cliente ; tale vincolo integra routing, flusso e assegnamento in un’unica struttura

for i in customers:
    model.addConstr(
        gp.quicksum(f[j, i] for j in nodes if j != i) -
        gp.quicksum(f[i, j] for j in nodes if j != i)
        ==
        1 - gp.quicksum(y[i, b] for b in lockers)
    )
 
#questo vincolo è relativo alla conservazione del flusso sui nodi locker (intesi come nodi di accumulo),il flusso entrante deve essere pari al 
#numero di clienti assegnati a quel locker ; in questo modo il veicolo deposita esattamente la quantità di merce necessaria e non scaricha pacchi in locker non utilizzati

for b in lockers:
    model.addConstr(
        gp.quicksum(f[j, b] for j in nodes if j != b) -
        gp.quicksum(f[b, j] for j in nodes if j != b)
        ==
        gp.quicksum(y[i, b] for i in customers)
    )
 
#il vincolo di capacità del veicolo impone che il flusso su un arco sia nullo se l’arco non viene percorso , nel caso in cui l'arco venga percorso 
#il flusso non deve superare la capacità del veicolo stesso

for i in nodes:
    for j in nodes:
        if i != j:
            model.addConstr(f[i, j] <= Q * x[i, j])
 
model.setParam('TimeLimit', 600) #limite (temporale) massimo del solver pari a 10'
model.setParam('Heuristics', 0) #questo comando disattiva le euristiche di Gurobi, il solver cercherà soluzione solo tramite Branch&Bound
model.optimize() #avvia l'ottimizzazione
 
# REPORT DEI RISULTATI FINALI , METODO ESATTO-ISTANZA GRANDE

print("\nREPORT FINALE PROGETTO: SOLUZIONE AMMISSIBILE (Time Limit)")
print(f"ISTANZA DA {n_c} CLIENTI - MAPPA 200x200")
print("=" * 50)   #intestazione del report, specifica che si tratta dell'istanza grande (100 clienti) con mappa 200x200
 
if model.SolCount > 0:  #controlla che il modello abbia trovato almeno una soluzione ammissibile entro il time limit
   
    obj_val = model.ObjVal  #valore della funzione obiettivo
    gap_residuo = model.MIPGap * 100  #gap in valore % tra il miglior bound teorico e la migior soluzione trovata
 
 
    # CALCOLO DELLA DISTANZA REALE

    pax_ai_locker = sum(y[i, b].X for i in customers for b in lockers) #numero totale di clienti assegnati ai locker (somma delle y)
    distanza_reale = obj_val - pax_ai_locker #distanza effettiva percorsa dai veicoli, ottenuta sottraendo dal valore obiettivo il contributo 
    #dei locker (coerente con la struttura dell’obiettivo)
 
    print(f"Valore Funzione Obiettivo: {obj_val:.2f}")
    print(f"Distanza reale percorsa dai furgoni: {distanza_reale:.2f}")
    print(f"Gap residuo: {gap_residuo:.2f}%")
 
   # CONTEGGIO DEI VEICOLI EFFETTIVAMENTE UTILIZZATI

    veicoli_usati = int(round(sum(x[0, j].X for j in nodes if j != 0))) #somma degli archi uscenti dal deposito (0→j) ,ogni arco attivo rappresenta un veicolo che parte
    print(f"\n>>> NUMERO DI VEICOLI IMPIEGATI: {veicoli_usati} <<<")
 
    # STATISTICHE CONSEGNE

    home_delivery = int(round(n_c - pax_ai_locker)) #tale differenza determina il numero dei clienti serviti a domicilio dal veicolo
    print("\nSTATISTICHE CONSEGNE:")
    print(f"  - Consegne a domicilio (Home Delivery): {home_delivery}")

    # DETTAGLIO PER LOCKER

    lockers_attivi = []
    for b in lockers:
        pax_b = int(round(sum(y[i, b].X for i in customers)))   #per ogni locker conta quanti clienti vi sono assegnati
        if pax_b > 0:
            print(f"  - Clienti al Locker {b}: {pax_b}")   #stampa solo i locker effettivamente utilizzati e li memorizza in lockers_attivi
            lockers_attivi.append(b)
 
    print("\nDETTAGLIO LOCKER APERTI:")
    for b in lockers_attivi:
        coord = coords[b]
        print(f"  Locker {b} posizionato in: ({coord[0]:.2f}, {coord[1]:.2f})") #per ogni locker attivo,stampa le coordinate (x,y) e ciò permette di collegare l'uso dei locker alla loro posizione geografica sulla mappa
 
else:
    print("Nessuna soluzione trovata nel tempo limite di 300 secondi.")
 
 #è il caso in cui il solver non trovi alcuna soluzione ammissibile entro il time limit


Set parameter Username
Set parameter LicenseID to value 2716163
Academic license - for non-commercial use only - expires 2026-09-30
Set parameter TimeLimit to value 600
Set parameter Heuristics to value 0
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 3 3200U with Radeon Vega Mobile Gfx, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Non-default parameters:
TimeLimit  600
Heuristics  0

Optimize a model with 15622 rows, 26738 columns and 229074 nonzeros
Model fingerprint: 0x63235987
Variable types: 12769 continuous, 13969 integer (13969 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [2e-01, 3e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+02]
Presolve removed 2496 rows and 1283 columns
Presolve time: 0.74s
Presolved: 13126 rows, 25455 columns, 87785 nonzeros
Variable types: 12656 continuous, 12799 integer (12